In [1]:
import os
import pandas as pd
import numpy as np  

In [ ]:
path='C:\\GitHub\Time-Series-Library\dataset\hpi\cropped_CAN.csv'
target = 'OT'

df_raw = pd.read_csv(path)

### Analyze the results ###

In [ ]:

import numpy as np

result_path = os.path.join(os.getcwd(), "results")
model_path = "long_term_forecast_can_80_20_20_iTransformer_custom_ftMS_sl8_ll2_pl2_dm128_nh8_el2_dl1_df128_expand2_dc4_fc3_ebtimeF_dtTrue_Exp_0"
path_result_model = os.path.join(result_path, model_path)

# traverse the result_path directory including subdirectories and read the .npy files
npy_files = []  
for root, dirs, files in os.walk(result_path):
    for file in files:
        if file.endswith('.npy'):
            npy_files.append(os.path.join(root, file)) 

for files in npy_files:
    data = np.load(files, allow_pickle=True)
    # Now 'data' contains the NumPy array stored in 'data.npy'
    print(f"Loaded data from {files}:")
    print(data.shape) 
    print(data)

In [ ]:
if os.path.exists(npy_file):
	data = np.load(npy_file)
	# Now 'data' contains the NumPy array stored in 'data.npy'
	print(data)
else:
	print(f"File not found: {npy_file}")

'nt'

In [ ]:
import os
import sys
# Go up two levels from current dir
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
# Add the parent directory to sys.path
sys.path.append(parent_dir)

import numpy as np
import pandas as pd
import glob
import re
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from utils.timefeatures import time_features
from data_provider.m4 import M4Dataset, M4Meta
from data_provider.uea import subsample, interpolate_missing, Normalizer
from sktime.datasets import load_from_tsfile_to_dataframe
import warnings
from utils.augmentation import run_augmentation_single

In [63]:
path='C:\\GitHub\Time-Series-Library\dataset\hpi\cropped_CAN.csv'

target = 'OT'
# assert flag in ['train', 'test', 'val']
flag = 'train'
seq_len = 80

type_map = {'train': 0, 'val': 1, 'test': 2}
set_type = type_map[flag]

df_raw = pd.read_csv(path)
print('DATA SHAPE: ', df_raw.shape)

cols = list(df_raw.columns)
cols.remove(target)
cols.remove('date')
df_raw = df_raw[['date'] + cols + [target]]
num_train = int(len(df_raw) * 0.7)
num_test = int(len(df_raw) * 0.2)
num_vali = len(df_raw) - num_train - num_test
print('NUM TRAIN: ', num_train, 'NUM TEST: ', num_test, 'NUM VALI: ', num_vali) 
border1s = [0, num_train - seq_len, len(df_raw) - num_test - seq_len]
border2s = [num_train, num_train + num_vali, len(df_raw)]
border1 = border1s[set_type]
border2 = border2s[set_type]
df_raw.head(10)


DATA SHAPE:  (132, 17)
NUM TRAIN:  92 NUM TEST:  26 NUM VALI:  14


,date,country_code,Nominal_house_price_indices,Price_to_income_ratio,Price_to_rent_ratio,Real_house_price_indices,Rent_prices,CPI,gdp_growth,gdp_per_capita,Gross_domestic_product,age_20-64_pop,total_pop,Interest_Rate,unemp_rate,avg_wage_usd_ppp,OT
0,19900215,2,35.223650,74.067230,51.966571,53.708477,67.783852,60.94285,0.779285,36984.1,1015714.7,16715402.00,27380370.25,12.852270,7.633333,12009.0940,35.223650
1,19900515,2,33.745865,72.740464,49.313518,51.003729,68.433782,61.52225,8.632000,36697.3,1011639.4,16776527.00,27483959.50,13.410710,7.666667,12009.0940,33.745865
2,19900815,2,34.675206,72.803204,50.215366,51.810905,69.055515,62.12799,7.396309,36271.1,1004389.3,16837652.00,27587548.75,13.062500,8.166667,12009.0940,34.675206
3,19901115,2,34.101265,70.515671,48.912408,50.300523,69.721609,63.04978,1.832448,35797.6,995446.1,16898777.00,27691138.00,11.963820,9.133333,12009.0940,34.101265
4,19910215,2,35.648283,73.152174,50.680234,51.513538,70.342202,64.86700,-1.742357,35222.8,981125.5,16949545.75,27777708.50,10.003130,10.166670,12054.4280,35.648283
5,19910515,2,37.134187,77.328598,52.401239,53.306977,70.867697,65.34106,-2.550798,35298.1,985834.5,17000314.50,27864279.00,8.781250,10.333330,12054.4280,37.134187
6,19910815,2,35.675758,73.149310,49.968805,50.845642,71.398685,65.73611,-1.717187,35208.0,987142.0,17051083.25,27950849.50,8.850000,10.433330,12054.4280,35.675758
7,19911115,2,35.497877,72.552999,49.356486,50.818418,71.924044,65.63076,-0.659956,35157.1,988876.6,17101852.00,28037420.00,7.503125,10.333330,12054.4280,35.497877
8,19920215,2,35.533720,73.313198,49.065307,50.566701,72.423934,65.89413,0.861166,35114.4,989574.7,17146579.25,28120881.00,6.965909,10.600000,12230.4015,35.533720
9,19920515,2,36.169130,73.427339,49.631857,51.271498,72.877505,66.23650,0.493783,35044.7,990702.4,17191306.50,28204342.00,6.039773,11.000000,12230.4015,36.169130


In [64]:
df_stamp = df_raw[['date']][border1:border2]
df_stamp

,date
0,19900215
1,19900515
2,19900815
3,19901115
4,19910215
...,...
87,20111115
88,20120215
89,20120515
90,20120815


In [65]:
# df_stamp['date'] = pd.to_datetime(df_stamp.date)
# df_stamp

In [66]:
df_stamp['date'] = pd.to_datetime(df_stamp['date'].astype(str), format='%Y%m%d')

In [67]:
df_stamp['month'] = df_stamp.date.apply(lambda row: row.month, 1)
df_stamp['day'] = df_stamp.date.apply(lambda row: row.day, 1)
df_stamp['weekday'] = df_stamp.date.apply(lambda row: row.weekday(), 1)
df_stamp['hour'] = df_stamp.date.apply(lambda row: row.hour, 1)
data_stamp = df_stamp.drop(['date'], 1).values
data_stamp

C:\Users\ghosh\AppData\Local\Temp\ipykernel_29040\1235507595.py:5: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  data_stamp = df_stamp.drop(['date'], 1).values


array([[ 2, 15,  3,  0],
       [ 5, 15,  1,  0],
       [ 8, 15,  2,  0],
       [11, 15,  3,  0],
       [ 2, 15,  4,  0],
       [ 5, 15,  2,  0],
       [ 8, 15,  3,  0],
       [11, 15,  4,  0],
       [ 2, 15,  5,  0],
       [ 5, 15,  4,  0],
       [ 8, 15,  5,  0],
       [11, 15,  6,  0],
       [ 2, 15,  0,  0],
       [ 5, 15,  5,  0],
       [ 8, 15,  6,  0],
       [11, 15,  0,  0],
       [ 2, 15,  1,  0],
       [ 5, 15,  6,  0],
       [ 8, 15,  0,  0],
       [11, 15,  1,  0],
       [ 2, 15,  2,  0],
       [ 5, 15,  0,  0],
       [ 8, 15,  1,  0],
       [11, 15,  2,  0],
       [ 2, 15,  3,  0],
       [ 5, 15,  2,  0],
       [ 8, 15,  3,  0],
       [11, 15,  4,  0],
       [ 2, 15,  5,  0],
       [ 5, 15,  3,  0],
       [ 8, 15,  4,  0],
       [11, 15,  5,  0],
       [ 2, 15,  6,  0],
       [ 5, 15,  4,  0],
       [ 8, 15,  5,  0],
       [11, 15,  6,  0],
       [ 2, 15,  0,  0],
       [ 5, 15,  5,  0],
       [ 8, 15,  6,  0],
       [11, 15,  0,  0],


In [61]:
df_stamp['year'] = df_stamp['date'].dt.year
df_stamp['quarter'] = df_stamp['date'].dt.quarter
df_stamp['month'] = df_stamp['date'].dt.month
data_stamp = df_stamp.drop(['date'], 1).values
data_stamp

C:\Users\ghosh\AppData\Local\Temp\ipykernel_29040\1341731085.py:4: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  data_stamp = df_stamp.drop(['date'], 1).values


array([[1990,    1,    2],
       [1990,    2,    5],
       [1990,    3,    8],
       [1990,    4,   11],
       [1991,    1,    2],
       [1991,    2,    5],
       [1991,    3,    8],
       [1991,    4,   11],
       [1992,    1,    2],
       [1992,    2,    5],
       [1992,    3,    8],
       [1992,    4,   11],
       [1993,    1,    2],
       [1993,    2,    5],
       [1993,    3,    8],
       [1993,    4,   11],
       [1994,    1,    2],
       [1994,    2,    5],
       [1994,    3,    8],
       [1994,    4,   11],
       [1995,    1,    2],
       [1995,    2,    5],
       [1995,    3,    8],
       [1995,    4,   11],
       [1996,    1,    2],
       [1996,    2,    5],
       [1996,    3,    8],
       [1996,    4,   11],
       [1997,    1,    2],
       [1997,    2,    5],
       [1997,    3,    8],
       [1997,    4,   11],
       [1998,    1,    2],
       [1998,    2,    5],
       [1998,    3,    8],
       [1998,    4,   11],
       [1999,    1,    2],
 

In [48]:
# df_stamp = df_raw[['date']][border1:border2]
# df_stamp
# df_stamp['date'] = pd.to_datetime(df_stamp.date)
# df_stamp
data_stamp = time_features(pd.to_datetime(df_stamp['date'].values), freq='M')
data_stamp
data_stamp = data_stamp.transpose(1, 0)
data_stamp

TypeError: 'DataFrame' object is not callable

In [39]:

df_raw['date'] = pd.to_datetime(df_raw['date'].astype(str), format='%Y%m%d')

# Extract time features
df_raw['year'] = df_raw['date'].dt.year
df_raw['quarter'] = df_raw['date'].dt.quarter
df_raw['month'] = df_raw['date'].dt.month

# Normalize/Scale time features
scaler = StandardScaler()
time_features = df_raw[['year', 'quarter', 'month']]
scaled_time_features = scaler.fit_transform(time_features)

# Add scaled features back to DataFrame
df_raw[['year_scaled', 'quarter_scaled', 'month_scaled']] = scaled_time_features

df_raw.head(200)


,date,country_code,Nominal_house_price_indices,Price_to_income_ratio,Price_to_rent_ratio,Real_house_price_indices,Rent_prices,CPI,gdp_growth,gdp_per_capita,...,Interest_Rate,unemp_rate,avg_wage_usd_ppp,OT,year,quarter,month,year_scaled,quarter_scaled,month_scaled
0,1990-02-15,2,35.223650,74.067230,51.966571,53.708477,67.783852,60.94285,0.779285,36984.1,...,12.852270,7.633333,12009.0940,35.223650,1990,1,2,-1.680336,-1.341641,-1.341641
1,1990-05-15,2,33.745865,72.740464,49.313518,51.003729,68.433782,61.52225,8.632000,36697.3,...,13.410710,7.666667,12009.0940,33.745865,1990,2,5,-1.680336,-0.447214,-0.447214
2,1990-08-15,2,34.675206,72.803204,50.215366,51.810905,69.055515,62.12799,7.396309,36271.1,...,13.062500,8.166667,12009.0940,34.675206,1990,3,8,-1.680336,0.447214,0.447214
3,1990-11-15,2,34.101265,70.515671,48.912408,50.300523,69.721609,63.04978,1.832448,35797.6,...,11.963820,9.133333,12009.0940,34.101265,1990,4,11,-1.680336,1.341641,1.341641
4,1991-02-15,2,35.648283,73.152174,50.680234,51.513538,70.342202,64.86700,-1.742357,35222.8,...,10.003130,10.166670,12054.4280,35.648283,1991,1,2,-1.575315,-1.341641,-1.341641
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,2021-11-15,2,166.404400,143.373008,151.363781,150.010244,109.940775,113.80040,2.198266,52100.0,...,0.160429,6.166667,16713.2885,166.404400,2021,4,11,1.575315,1.341641,1.341641
128,2022-02-15,2,175.721232,146.825347,157.273144,155.697875,111.734072,116.14430,-1.058509,52382.0,...,0.407757,5.766667,16516.4510,175.721232,2022,1,2,1.680336,-1.341641,-1.341641
129,2022-05-15,2,181.008142,150.075377,159.814953,157.394062,113.265242,119.72610,6.321567,52714.0,...,1.428814,5.166667,16516.4510,181.008142,2022,2,5,1.680336,-0.447214,-0.447214
130,2022-08-15,2,174.379185,144.390968,152.191923,150.638003,114.582685,120.72690,7.375694,52687.8,...,3.101768,5.033333,16516.4510,174.379185,2022,3,8,1.680336,0.447214,0.447214


In [ ]:
df_raw.dtypes

date                             int64
country_code                     int64
Nominal_house_price_indices    float64
Price_to_income_ratio          float64
Price_to_rent_ratio            float64
Real_house_price_indices       float64
Rent_prices                    float64
CPI                            float64
gdp_growth                     float64
gdp_per_capita                 float64
Gross_domestic_product         float64
age_20-64_pop                  float64
total_pop                      float64
Interest_Rate                  float64
unemp_rate                     float64
avg_wage_usd_ppp               float64
OT                             float64
dtype: object